# Cedar | 3D OPFData: PHD versus Algorithm 1

**Task.** Fit CART on CSV columns 1–3 to predict cost (column 0), reconstruct a Hessian using Algorithm 1, and independently compute residual-based PHD. This notebook imports the **same estimator** used in the minimal Colab quadratic example. Neither matrix is known to equal the true OPF Hessian.

**Algorithm 1.** From each internal split, record its child-node mean responses, contrast \(\gamma_A=2(\bar y_R-\bar y_L)/(u_s-\ell_s)\), and quadratic effective location \(z_A\). Form all unordered pairs of nodes splitting the same variable; use \((z_B-z_A)^\top\) as design rows and \(\gamma_B-\gamma_A\) as responses. If a design matrix has rank < 3, return **not identified**. Otherwise solve each Hessian row by least squares and return \((\widetilde Q+\widetilde Q^\top)/2\). Keep the raw rows for diagnostics.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from algorithm1_tree_hessian import estimate_tree_hessian

# If running elsewhere, edit or place the CSV at one of these paths.
CSV_PATH = next((p for p in [Path('/mnt/data/118_ieee_smol.csv'),
    Path('/Users/tonys/Desktop/Class/Decision Trees/datasets/118_ieee_smol.csv'),
    Path('/content/drive/MyDrive/118_ieee_smol.csv'), Path('118_ieee_smol.csv')]
    if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError('Set CSV_PATH to 118_ieee_smol.csv')

D = 3
frame = pd.read_csv(CSV_PATH, header=None, usecols=range(D + 1))
y, X = frame.iloc[:, 0].to_numpy(float), frame.iloc[:, 1:].to_numpy(float)
print(f'{len(y)} observations, {D} selected demands; columns 1...{D}')


5000 observations, 3 selected demands; columns 1...3


In [1]:
def phd(X, y):
    """Residual-based PHD; not a known true Hessian on OPFData."""
    z = X - X.mean(0)
    D = np.c_[np.ones(len(y)), z]
    r = y - D @ np.linalg.lstsq(D, y, rcond=None)[0]
    S, M = z.T @ z / len(y), z.T @ (r[:, None] * z) / len(y)
    A = np.linalg.solve(S, M)
    H = np.linalg.solve(S, A.T).T
    return (H + H.T) / 2

tree = DecisionTreeRegressor(max_depth=7, random_state=0).fit(X, y)
Qhat, Qraw, diagnostics = estimate_tree_hessian(tree, X)
if Qhat is None:
    raise ValueError(f'Algorithm 1: not identified. Rows: {diagnostics}')
Hphd = phd(X, y)
print('Rows (axis, internal nodes, pairs, rank):', diagnostics)
print('Tree raw rows:\n', np.round(Qraw, 2))
print('Algorithm 1 final symmetric Hessian:\n', np.round(Qhat, 2))
print('Residual PHD matrix:\n', np.round(Hphd, 2))
print('Raw asymmetry max:', round(np.max(abs(Qraw-Qraw.T)), 2))


Rows (axis, internal nodes, pairs, rank): [(1, 44, 946, 3), (2, 41, 820, 3), (3, 39, 741, 3)]
Tree raw rows:
 [[ -399605.64  1153309.03  1158807.55]
 [-1022674.04 -1601111.01   120826.5 ]
 [  777750.47 -2121290.08   315482.4 ]]
Algorithm 1 final symmetric Hessian:
 [[ -399605.64    65317.49   968279.01]
 [   65317.49 -1601111.01 -1000231.79]
 [  968279.01 -1000231.79   315482.4 ]]
Residual PHD matrix:
 [[ 152100.92 -232667.53 -106877.38]
 [-232667.53  956292.5  -353929.23]
 [-106877.38 -353929.23  249668.28]]
Raw asymmetry max: 2242116.58


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, H, label in zip(axes, [Qraw, Qhat, Hphd], ['Tree: raw rows', 'Tree: Algorithm 1 output', 'Residual PHD']):
    a = H / 1e6
    ax.imshow(a, cmap='RdBu_r', vmin=-max(abs(a).max(), 1e-8), vmax=max(abs(a).max(), 1e-8))
    for i, j in np.ndindex(H.shape):
        ax.text(j, i, f'{a[i,j]:+.2f}', ha='center', va='center', fontsize=9)
    ax.set(title=label, xticks=range(X.shape[1]), yticks=range(X.shape[1]))
fig.suptitle('Entries / 1,000,000 (original units)')
plt.tight_layout()
plt.show()


**Interpretation.** Raw off-diagonal discrepancies describe internal inconsistency, not error against the unknown true OPF Hessian. Symmetrization does not fix faulty derivative contrasts. The geometric effective-location identity holds for a quadratic with uniform regional means; empirical CART child-node mean responses may violate those assumptions. Dataset extrema are provisional outer-box boundaries.